# QLoRA on a free Kaggle T4

Account `notsaaadkhan`. Run All. This notebook trains `Qwen/Qwen2.5-1.5B-Instruct` on the full Spider train split with `configs/train_qlora_kaggle.yaml` (2 epochs, `max_steps` 900 as a ceiling). Plan on about 3-4 GPU hours. Stop the session if the Kaggle timer passes 4 hours. That duration is a budget, not a measurement from a finished run.

Accelerator: GPU T4. Internet: On. Dataset: the private `localllm-ops` upload. `HF_TOKEN` is optional. The base model is public. Click path is in `RUNBOOK.md`.

In [ ]:
import os
import shutil
from pathlib import Path

candidates = [
    Path.cwd(),
    Path("/kaggle/input/localllm-ops"),
    Path("/kaggle/input/datasets/notsaaadkhan/localllm-ops"),
]
found = None
for path in candidates:
    if (path / "pyproject.toml").is_file() and (path / "src" / "localllm").is_dir():
        found = path
        break
if found is None and Path("/kaggle/input").exists():
    for match in Path("/kaggle/input").glob("**/pyproject.toml"):
        if (match.parent / "src" / "localllm").is_dir():
            found = match.parent
            break
if found is None:
    raise SystemExit(
        "Upload the repo zip as a private Kaggle dataset named localllm-ops. See RUNBOOK.md."
    )
working = Path("/kaggle/working")
if working.is_dir() and found.resolve() != (working / "localllm-ops").resolve():
    dest = working / "localllm-ops"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(
        found,
        dest,
        ignore=shutil.ignore_patterns(".git", ".venv", "data", "repo.bundle"),
    )
    found = dest
os.chdir(found)
print("repo", Path.cwd())


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        print("HF_TOKEN loaded from Kaggle secrets")
    else:
        print("HF_TOKEN secret is empty. The public Qwen model does not need it.")
except Exception:
    print("No HF_TOKEN. Qwen/Qwen2.5-1.5B-Instruct is public, so training continues.")


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[train]"])


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "localllm.data.prep_spider",
        "--out",
        "data/spider",
        "--download-sqlite",
        "--require-sqlite",
    ]
)
manifest = json.loads(Path("data/spider/manifest.json").read_text(encoding="utf-8"))
counts = manifest["counts"]
print(counts)
print("sqlite_missing", manifest["sqlite_missing"])
assert counts["train"] == 6650, counts
assert counts["val"] == 350, counts
assert counts["test"] == 1034, counts
assert manifest["sqlite_missing"] == []


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "localllm.train.sft",
        "--config",
        "configs/train_qlora_kaggle.yaml",
        "--run",
    ]
)
print("adapter", Path("outputs/qlora-kaggle").resolve())
